In [ ]:
# import libraries
import numpy as np
import pandas as pd

In [ ]:
# load data
df = pd.read_csv(r"C:\Users\shali\OneDrive\Desktop\FinanceX\data\FinanceX_transaction.csv")
df.head()

,customer_id,transaction_date,account_type,transaction_amount,transaction_direction,account_balance,merchant_category,emi_amount,transaction_status,channel,transaction_hour
0,CUST015796,2019-01-01,Current,1820.17,Debit,609365.31,Food & Dining,0.00,Success,Branch,15
1,CUST000861,2019-01-01,Current,392.67,Credit,14451.14,Education,0.00,Success,Mobile_App,3
2,CUST076821,2019-01-01,Fixed Deposit,1255.97,Debit,47621.87,Utilities,1433.91,Success,Mobile_App,18
3,CUST054887,2019-01-01,Savings,2580.68,Debit,34467.85,Travel,6280.35,Success,API,8
4,CUST006266,2019-01-01,Fixed Deposit,2573.80,Debit,26617.40,Utilities,0.00,Success,API,14


In [ ]:
# take dataframe which transaction_status = success
reg_df = df[df["transaction_status"] == "Success"].copy()
print("Shape:", reg_df.shape)

Shape: (506247, 11)


In [ ]:
# transaction_date convert to datetime then month extract
reg_df['transaction_date'] = pd.to_datetime(reg_df['transaction_date'])
reg_df['month'] = reg_df['transaction_date'].dt.to_period('M')
reg_df[["transaction_date", "month"]].head()

,transaction_date,month
0,2019-01-01,2019-01
1,2019-01-01,2019-01
2,2019-01-01,2019-01
3,2019-01-01,2019-01
4,2019-01-01,2019-01


In [ ]:
# debit_amount and credit_amount feature creation
reg_df["debit_amount"] = (
    reg_df["transaction_amount"].where(reg_df["transaction_direction"] == "Debit",0)
)

reg_df["credit_amount"] = (
    reg_df["transaction_amount"].where(reg_df["transaction_direction"] == "Credit",0)
)

In [ ]:
# monthly aggregation
monthly = (
    reg_df
    .groupby(
        ["customer_id", "month"],
        as_index=False
    )
    .agg(
        monthly_expense=("debit_amount", "sum"),
        monthly_income=("credit_amount", "sum"),
        transaction_count=("transaction_amount", "count")
    )
)

monthly.head()

,customer_id,month,monthly_expense,monthly_income,transaction_count
0,CUST000001,2020-03,892.86,0.00,1
1,CUST000001,2021-05,0.00,328.91,1
2,CUST000001,2021-08,1000.00,0.00,1
3,CUST000001,2023-02,584.18,0.00,1
4,CUST000001,2023-12,470.63,0.00,1


In [ ]:
# monthly savings columns creation
monthly["monthly_savings"] = (
    monthly["monthly_income"]
    - monthly["monthly_expense"]
)

In [ ]:
# average transaction amount calculation
monthly["avg_transaction_amount"] = (
    (
        monthly["monthly_income"]
        + monthly["monthly_expense"]
    )
    / monthly["transaction_count"]
)

In [ ]:
# sorting the monthly dataframe by customer_id and month
monthly = (
    monthly
    .sort_values(
        ["customer_id", "month"]
    )
    .reset_index(drop=True)
)

monthly.head()

,customer_id,month,monthly_expense,monthly_income,transaction_count,monthly_savings,avg_transaction_amount
0,CUST000001,2020-03,892.86,0.00,1,-892.86,892.86
1,CUST000001,2021-05,0.00,328.91,1,328.91,328.91
2,CUST000001,2021-08,1000.00,0.00,1,-1000.00,1000.00
3,CUST000001,2023-02,584.18,0.00,1,-584.18,584.18
4,CUST000001,2023-12,470.63,0.00,1,-470.63,470.63


In [ ]:
# creating previous month expense column
monthly["previous_month_expense"] = (
    monthly
    .groupby("customer_id")["monthly_expense"]
    .shift(1)
)

Previous month expense created.
400624


In [ ]:
# creating 1–6 month historical lags
for lag in range(1, 7):
    monthly[f"expense_lag_{lag}"] = (
        monthly
        .groupby("customer_id")["monthly_expense"]
        .shift(lag)
    )

1–6 month historical lags created.


In [ ]:
# creating 3 month average expense column
monthly["3_month_avg_expense"] = (
    monthly[
        [
            "expense_lag_1",
            "expense_lag_2",
            "expense_lag_3"
        ]
    ].mean(axis=1)
)

In [ ]:
# creating 6 month average expense column
monthly["6_month_avg_expense"] = (
    monthly[
        [
            "expense_lag_1",
            "expense_lag_2",
            "expense_lag_3",
            "expense_lag_4",
            "expense_lag_5",
            "expense_lag_6"
        ]
    ].mean(axis=1)
)

In [ ]:
# creating expense trend column
monthly["expense_trend"] = (
    monthly["previous_month_expense"]
    - monthly["3_month_avg_expense"]
)

In [ ]:
# creating next month target column
monthly["next_month_expense"] = (
    monthly
    .groupby("customer_id")["monthly_expense"]
    .shift(-1)
)

Next month target created.


In [ ]:
required_columns = [
    "previous_month_expense",
    "expense_lag_1",
    "expense_lag_2",
    "expense_lag_3",
    "expense_lag_4",
    "expense_lag_5",
    "expense_lag_6",
    "next_month_expense"
]

regression_dataset = (
    monthly
    .dropna(subset=required_columns)
    .copy()
)

regression_dataset.shape

Regression dataset shape: (41721, 18)


In [ ]:
regression_columns = [
    "monthly_expense",
    "previous_month_expense",
    "3_month_avg_expense",
    "6_month_avg_expense",
    "monthly_income",
    "monthly_savings",
    "transaction_count",
    "avg_transaction_amount",
    "expense_trend",
    "next_month_expense"
]

regression_dataset = (regression_dataset[regression_columns].reset_index(drop=True))

In [ ]:
regression_dataset.shape, regression_dataset.columns.tolist(), regression_dataset.isna().sum()

Shape: (41721, 10)

Columns:
['monthly_expense', 'previous_month_expense', '3_month_avg_expense', '6_month_avg_expense', 'monthly_income', 'monthly_savings', 'transaction_count', 'avg_transaction_amount', 'expense_trend', 'next_month_expense']

Missing Values:
monthly_expense           0
previous_month_expense    0
3_month_avg_expense       0
6_month_avg_expense       0
monthly_income            0
monthly_savings           0
transaction_count         0
avg_transaction_amount    0
expense_trend             0
next_month_expense        0
dtype: int64


In [208]:
regression_dataset.head()

,monthly_expense,previous_month_expense,3_month_avg_expense,6_month_avg_expense,monthly_income,monthly_savings,transaction_count,avg_transaction_amount,expense_trend,next_month_expense
0,83473.69,87741.54,30142.893333,18574.110000,0.00,-83473.69,1,83473.69,57598.646667,2988.95
1,2988.95,83473.69,57738.410000,31637.840000,0.00,-2988.95,1,2988.95,25735.280000,1190.77
2,1190.77,2988.95,58068.060000,30542.456667,0.00,-1190.77,1,1190.77,-55079.110000,0.00
3,0.00,569.06,739.546667,598.828333,1995673.07,1995673.07,1,1995673.07,-170.486667,19871.64
4,1535.78,56672.35,19016.513333,11914.645000,0.00,-1535.78,1,1535.78,37655.836667,5205.81
